# Generate Bracket Visualization

Generates color-coded bracket images (PNG and interactive HTML) for any completed
or upcoming season using the ELO + XGBoost pipeline.

**For a past season:** set `DATA_DIR = "../data/2026"` (cumulative, has all seeds).
**For current season submission:** set `DATA_DIR = "../data/{CURRENT_SEASON}"`.

Output: `output/{SEASON}/{METHOD}/{GENDER}/bracket.png` and `bracket.html`

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================
SEASON = 2026
DATA_DIR = f"../data/{SEASON}"  # season-specific: no future results leaked
# For scoring/backtesting past seasons, use DATA_DIR = "../data/2026" (full history)
METHOD = "elo_enhanced"     # 'elo', 'elo_enhanced', or 'ensemble'
GENDER = "M"                # 'M' or 'W'
# =============================================================================

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('..'))

from src.data_classes.processing.DataManager import MarchMadnessDataManager
from src.data_classes.processing.EloRatingSystem import EloRatingSystem
from src.data_classes.processing.TeamStatsCalculator import TeamStatsCalculator
from src.data_classes.processing.MLModel import MarchMadnessMLModel
from src.data_classes.processing.Predictor import MarchMadnessPredictor
from src.visualization.bracket_viz import visualize_bracket, visualize_bracket_html

def make_output_path(year, method, gender, filename):
    path = f"../output/{year}/{method}/{gender}"
    os.makedirs(path, exist_ok=True)
    return f"{path}/{filename}"

print(f"Season: {SEASON}, Gender: {GENDER}, Method: {METHOD}")
print(f"Data: {DATA_DIR}")

In [ ]:
# Initialize and train the ELO + ML pipeline
data_manager = MarchMadnessDataManager(data_dir=DATA_DIR, gender=GENDER)
data_manager.load_data()

elo = EloRatingSystem(data_manager=data_manager)
stats = TeamStatsCalculator(data_manager=data_manager)
ml = MarchMadnessMLModel(data_manager=data_manager, elo_system=elo, stats_calculator=stats)

predictor = MarchMadnessPredictor(
    data_manager=data_manager,
    elo_system=elo,
    stats_calculator=stats,
    ml_model=ml,
    current_season=SEASON,
)
predictor.initialize_models()

print("Pipeline ready.")

In [ ]:
# Generate bracket PNG
png_path = make_output_path(SEASON, METHOD, GENDER, "bracket.png")

fig = visualize_bracket(
    predictor=predictor,
    season=SEASON,
    method=METHOD,
    output_path=png_path,
)

print(f"Saved bracket PNG: {png_path}")

# Display inline
from IPython.display import display
display(fig)
plt.close(fig)

In [ ]:
# Generate interactive HTML bracket
html_path = make_output_path(SEASON, METHOD, GENDER, "bracket.html")

visualize_bracket_html(
    predictor=predictor,
    season=SEASON,
    method=METHOD,
    output_path=html_path,
)

print(f"Saved interactive bracket HTML: {html_path}")
print(f"Open in browser: file://{os.path.abspath(html_path)}")

In [ ]:
# Women's bracket (repeat with gender='W')
# Requires women's data in DATA_DIR
GENDER_W = "W"

try:
    dm_w = MarchMadnessDataManager(data_dir=DATA_DIR, gender=GENDER_W)
    dm_w.load_data()

    elo_w = EloRatingSystem(data_manager=dm_w)
    stats_w = TeamStatsCalculator(data_manager=dm_w)
    ml_w = MarchMadnessMLModel(data_manager=dm_w, elo_system=elo_w, stats_calculator=stats_w)

    predictor_w = MarchMadnessPredictor(
        data_manager=dm_w,
        elo_system=elo_w,
        stats_calculator=stats_w,
        ml_model=ml_w,
        current_season=SEASON,
    )
    predictor_w.initialize_models()

    png_path_w = make_output_path(SEASON, METHOD, GENDER_W, "bracket.png")
    html_path_w = make_output_path(SEASON, METHOD, GENDER_W, "bracket.html")

    fig_w = visualize_bracket(
        predictor=predictor_w,
        season=SEASON,
        method=METHOD,
        output_path=png_path_w,
    )
    visualize_bracket_html(
        predictor=predictor_w,
        season=SEASON,
        method=METHOD,
        output_path=html_path_w,
    )
    from IPython.display import display
    display(fig_w)
    plt.close(fig_w)
    print(f"Women's brackets saved: {png_path_w}, {html_path_w}")

except (FileNotFoundError, Exception) as e:
    print(f"Women's data not found in {DATA_DIR} — skipping women's bracket")
    print(f"  Reason: {e}")
    print(f"  Tip: Use DATA_DIR = '../data/2026' for complete data including women's.")

# Historical Archive Mode

Visualize a past submission vs actual tournament results.

Set `ARCHIVE_MODEL` to the CSV name (without `.csv`) from `archive/submissions/{ARCHIVE_SEASON}/`.

In [ ]:
# =============================================================================
# HISTORICAL ARCHIVE MODE — visualize a past submission vs actual results
# =============================================================================
# Set ARCHIVE_MODEL to the CSV name (without .csv) from archive/submissions/{SEASON}/
ARCHIVE_MODEL = "xgb_ensemble_clean"   # e.g. "elo_pure", "baseline_massey"
ARCHIVE_SEASON = 2025
ARCHIVE_GENDER = "M"

import warnings
warnings.filterwarnings('ignore')
import pandas as pd

from src.bracket_analysis import SimplePredictor
from src.visualization.bracket_viz import visualize_historical_bracket

# Load data/2026 (cumulative) so we have actual tournament results for ARCHIVE_SEASON.
# current_season=ARCHIVE_SEASON+1 suppresses the leakage warning because
# max_tourney_season=2025 < current_season=2026 — this is intentional.
archive_dm = MarchMadnessDataManager(
    data_dir="../data/2026", gender=ARCHIVE_GENDER, current_season=ARCHIVE_SEASON + 1
)
archive_dm.load_data()

# Load the archive submission CSV
sub_df = pd.read_csv(f"../archive/submissions/{ARCHIVE_SEASON}/{ARCHIVE_MODEL}.csv")

# Build SimplePredictor — pass tourney_results so simulate_historical_bracket() works
predictor_arch = SimplePredictor(
    teams_df=archive_dm.data["teams"],
    seeds_df=archive_dm.data["tourney_seeds"],
    slots_df=archive_dm.data["tourney_slots"],
    predictions_df=sub_df,
    results_df=archive_dm.data["tourney_results"],
    current_season=ARCHIVE_SEASON,
)

# Generate and save the historical bracket
arch_output = f"../archive/brackets/{ARCHIVE_SEASON}/{ARCHIVE_MODEL}/{ARCHIVE_GENDER}/bracket.png"
os.makedirs(os.path.dirname(arch_output), exist_ok=True)

fig_arch, metrics_arch = visualize_historical_bracket(
    predictor=predictor_arch,
    season=ARCHIVE_SEASON,
    method=ARCHIVE_MODEL,
    output_path=arch_output,
)
print(f"Accuracy: {metrics_arch['accuracy']:.1%} ({metrics_arch['correct']}/{metrics_arch['total']} games)")
print(f"Saved: {arch_output}")
display(fig_arch)
plt.close(fig_arch)